# Week 77, Day 5: Deployment

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w077d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('miniflask.py', 'w', encoding='utf-8') as f:
    f.write('"""A small subset of Flask that runs in the page (Week 77).\n\nThe names and behaviour follow Flask\'s own, so code written against this module also runs with\nthe real Flask (pip install flask): Flask, request, g, jsonify, abort, redirect, make_response,\nurl_for, Blueprint, and app.test_client(). It\'s a teaching copy, not the real thing: no templates,\nsessions, streaming or debugger.\n"""\nimport io\nimport json\nimport re\nfrom contextvars import ContextVar\nfrom http import HTTPStatus\nfrom urllib.parse import parse_qs, urlencode, urlsplit\n\n_current = ContextVar("request")\n\n\nclass HTTPException(Exception):\n    def __init__(self, code, description=None):\n        self.code = code\n        self.name = HTTPStatus(code).phrase\n        self.description = description or self.name\n        super().__init__(f"{code} {self.name}: {self.description}")\n\n\ndef abort(code, description=None):\n    raise HTTPException(code, description)\n\n\nclass Headers:\n    """Case-insensitive headers."""\n    def __init__(self, items=()):\n        self._items = [(k, str(v)) for k, v in (items.items() if isinstance(items, dict) else items)]\n\n    def get(self, name, default=None):\n        for k, v in self._items:\n            if k.lower() == name.lower():\n                return v\n        return default\n\n    def __getitem__(self, name):\n        value = self.get(name)\n        if value is None:\n            raise KeyError(name)\n        return value\n\n    def __contains__(self, name):\n        return self.get(name) is not None\n\n    def __setitem__(self, name, value):\n        self._items = [(k, v) for k, v in self._items if k.lower() != name.lower()] + [(name, str(value))]\n\n    def items(self):\n        return list(self._items)\n\n\nclass Args(dict):\n    """Query or form values: .get(name) gives the first value; .getlist(name) all of them."""\n    def __init__(self, pairs):\n        super().__init__()\n        self._lists = {}\n        for k, v in pairs:\n            self._lists.setdefault(k, []).append(v)\n            dict.setdefault(self, k, v)\n\n    def get(self, key, default=None, type=None):\n        if key not in self:\n            return default\n        value = self[key]\n        if type is not None:\n            try:\n                return type(value)\n            except (TypeError, ValueError):\n                return default\n        return value\n\n    def getlist(self, key):\n        return list(self._lists.get(key, []))\n\n\nclass Request:\n    def __init__(self, environ):\n        self.environ = environ\n        self.method = environ["REQUEST_METHOD"].upper()\n        self.path = environ.get("PATH_INFO", "/") or "/"\n        self.args = Args(_pairs(environ.get("QUERY_STRING", "")))\n        items = [(k[5:].replace("_", "-").title(), v) for k, v in environ.items() if k.startswith("HTTP_")]\n        if environ.get("CONTENT_TYPE"):\n            items.append(("Content-Type", environ["CONTENT_TYPE"]))\n        self.headers = Headers(items)\n        length = int(environ.get("CONTENT_LENGTH") or 0)\n        self.data = environ["wsgi.input"].read(length) if length else b""\n        self.content_type = environ.get("CONTENT_TYPE", "")\n        self.form = Args(_pairs(self.data.decode("utf-8"))) if self.content_type.startswith("application/x-www-form-urlencoded") else Args([])\n        self.view_args = {}\n        cookie = environ.get("HTTP_COOKIE", "")\n        self.cookies = dict(p.strip().split("=", 1) for p in cookie.split(";") if "=" in p)\n\n    @property\n    def is_json(self):\n        return self.content_type.split(";")[0].strip() == "application/json"\n\n    def get_json(self, silent=False, force=False):\n        if not (self.is_json or force):\n            if silent:\n                return None\n            abort(415, "expected a JSON body (Content-Type: application/json)")\n        try:\n            return json.loads(self.data or b"null")\n        except ValueError:\n            if silent:\n                return None\n            abort(400, "the body isn\'t valid JSON")\n\n    @property\n    def json(self):\n        return self.get_json()\n\n\ndef _pairs(text):\n    return [(k, v) for k, vs in parse_qs(text, keep_blank_values=True).items() for v in vs]\n\n\nclass _Proxy:\n    def __init__(self, getter):\n        object.__setattr__(self, "_getter", getter)\n\n    def __getattr__(self, name):\n        return getattr(self._getter(), name)\n\n    def __setattr__(self, name, value):\n        setattr(self._getter(), name, value)\n\n\ndef _request():\n    try:\n        return _current.get()[0]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\ndef _g():\n    try:\n        return _current.get()[1]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\nclass _G:\n    def get(self, name, default=None):\n        return self.__dict__.get(name, default)\n\n\nrequest = _Proxy(_request)\ng = _Proxy(_g)\n\n\nclass Response:\n    def __init__(self, response=b"", status=200, headers=None, mimetype=None, content_type=None):\n        if isinstance(response, str):\n            response = response.encode("utf-8")\n        self.data = response\n        self.status_code = status\n        self.headers = Headers(headers or {})\n        if content_type or mimetype or "Content-Type" not in self.headers:\n            ctype = content_type or (f"{mimetype}; charset=utf-8" if mimetype and mimetype.startswith("text/") else mimetype) or "text/html; charset=utf-8"\n            self.headers["Content-Type"] = ctype\n\n    @property\n    def status(self):\n        return f"{self.status_code} {HTTPStatus(self.status_code).phrase}"\n\n    @property\n    def text(self):\n        return self.data.decode("utf-8")\n\n    def get_json(self):\n        return json.loads(self.data)\n\n    @property\n    def json(self):\n        return self.get_json()\n\n    def set_cookie(self, key, value="", max_age=None, path="/", secure=False, httponly=False, samesite=None):\n        parts = [f"{key}={value}"]\n        if max_age is not None:\n            parts.append(f"Max-Age={max_age}")\n        parts.append(f"Path={path}")\n        if secure:\n            parts.append("Secure")\n        if httponly:\n            parts.append("HttpOnly")\n        if samesite:\n            parts.append(f"SameSite={samesite}")\n        self.headers._items.append(("Set-Cookie", "; ".join(parts)))\n\n\ndef jsonify(*args, **kwargs):\n    data = args[0] if len(args) == 1 else (list(args) if args else kwargs)\n    return Response(json.dumps(data), headers={"Content-Type": "application/json"})\n\n\ndef redirect(location, code=302):\n    return Response(b"", code, {"Location": location})\n\n\ndef make_response(*args):\n    rv = args[0] if len(args) == 1 else args\n    return _App._to_response(rv)\n\n\ndef url_for(endpoint, **values):\n    return _current.get()[2].url_for(endpoint, **values)\n\n\n_CONVERTERS = {"int": (r"\\d+", int), "float": (r"\\d+(?:\\.\\d+)?", float), "path": (r".+", str), "string": (r"[^/]+", str)}\n\n\nclass _Rule:\n    def __init__(self, rule, endpoint, view, methods):\n        self.rule, self.endpoint, self.view = rule, endpoint, view\n        self.methods = {m.upper() for m in methods} | ({"HEAD"} if "GET" in {m.upper() for m in methods} else set())\n        regex, self.converters = "", {}\n        for piece in re.split(r"(<[^>]+>)", rule):\n            m = re.fullmatch(r"<(?:(\\w+):)?(\\w+)>", piece)\n            if m:\n                kind, name = m.group(1) or "string", m.group(2)\n                pattern, convert = _CONVERTERS[kind]\n                regex += f"(?P<{name}>{pattern})"\n                self.converters[name] = convert\n            else:\n                regex += re.escape(piece)\n        self.regex = re.compile(regex)\n\n    def match(self, path):\n        m = self.regex.fullmatch(path)\n        if not m:\n            return None\n        return {k: self.converters[k](v) for k, v in m.groupdict().items()}\n\n    def build(self, values):\n        def fill(m):\n            return str(values.pop(m.group(2)))\n        path = re.sub(r"<(?:(\\w+):)?(\\w+)>", fill, self.rule)\n        return path + (f"?{urlencode(values)}" if values else "")\n\n\nclass _Setup:\n    def route(self, rule, methods=("GET",), endpoint=None):\n        def decorate(view):\n            self._add(rule, endpoint or view.__name__, view, methods)\n            return view\n        return decorate\n\n    def get(self, rule, **kw):\n        return self.route(rule, methods=["GET"], **kw)\n\n    def post(self, rule, **kw):\n        return self.route(rule, methods=["POST"], **kw)\n\n    def put(self, rule, **kw):\n        return self.route(rule, methods=["PUT"], **kw)\n\n    def patch(self, rule, **kw):\n        return self.route(rule, methods=["PATCH"], **kw)\n\n    def delete(self, rule, **kw):\n        return self.route(rule, methods=["DELETE"], **kw)\n\n\nclass Blueprint(_Setup):\n    def __init__(self, name, import_name, url_prefix=""):\n        self.name, self.url_prefix = name, url_prefix\n        self.deferred = []\n\n    def _add(self, rule, endpoint, view, methods):\n        self.deferred.append((rule, endpoint, view, methods))\n\n\nclass _App(_Setup):\n    def __init__(self, import_name):\n        self.import_name = import_name\n        self.config = {"TESTING": False}\n        self.rules = []\n        self.before_funcs, self.after_funcs, self.error_handlers = [], [], {}\n\n    def _add(self, rule, endpoint, view, methods):\n        self.rules.append(_Rule(rule, endpoint, view, methods))\n\n    def register_blueprint(self, bp, url_prefix=None):\n        prefix = url_prefix if url_prefix is not None else bp.url_prefix\n        for rule, endpoint, view, methods in bp.deferred:\n            self._add(prefix + rule, f"{bp.name}.{endpoint}", view, methods)\n\n    def before_request(self, f):\n        self.before_funcs.append(f)\n        return f\n\n    def after_request(self, f):\n        self.after_funcs.append(f)\n        return f\n\n    def errorhandler(self, code):\n        def decorate(f):\n            self.error_handlers[code] = f\n            return f\n        return decorate\n\n    def url_for(self, endpoint, **values):\n        for r in self.rules:\n            if r.endpoint == endpoint:\n                return r.build(dict(values))\n        raise LookupError(f"no endpoint {endpoint!r}")\n\n    @staticmethod\n    def _to_response(rv):\n        status, headers = None, None\n        if isinstance(rv, tuple):\n            if len(rv) == 3:\n                rv, status, headers = rv\n            elif isinstance(rv[1], (dict, list)):\n                rv, headers = rv\n            else:\n                rv, status = rv\n        if isinstance(rv, Response):\n            response = rv\n        elif isinstance(rv, (dict, list)):\n            response = jsonify(rv)\n        elif isinstance(rv, (str, bytes)):\n            response = Response(rv)\n        elif rv is None:\n            raise TypeError("the view function did not return a valid response")\n        else:\n            raise TypeError(f"can\'t turn {type(rv).__name__} into a response")\n        if status is not None:\n            response.status_code = status\n        for k, v in (headers.items() if isinstance(headers, dict) else headers or []):\n            response.headers[k] = v\n        return response\n\n    def _error(self, e):\n        handler = self.error_handlers.get(e.code)\n        if handler:\n            return self._to_response(handler(e))\n        body = f"<h1>{e.code} {e.name}</h1><p>{e.description}</p>"\n        return Response(body, e.code)\n\n    def _dispatch(self, req):\n        path = req.path\n        allowed = set()\n        for r in self.rules:\n            values = r.match(path)\n            if values is None:\n                continue\n            if req.method in r.methods:\n                req.view_args = values\n                return r.view(**values)\n            allowed |= r.methods\n        if allowed:\n            raise _MethodNotAllowed(sorted(allowed | {"OPTIONS"}))\n        abort(404, "The requested URL was not found on the server.")\n\n    def full_dispatch(self, environ):\n        req = Request(environ)\n        token = _current.set((req, _G(), self))\n        try:\n            try:\n                rv = None\n                for f in self.before_funcs:\n                    rv = f()\n                    if rv is not None:\n                        break\n                if rv is None:\n                    rv = self._dispatch(req)\n                response = self._to_response(rv)\n            except _MethodNotAllowed as e:\n                response = self._error(HTTPException(405))\n                response.headers["Allow"] = ", ".join(e.allowed)\n            except HTTPException as e:\n                response = self._error(e)\n            except Exception as e:\n                if self.config.get("TESTING") and 500 not in self.error_handlers:\n                    raise\n                response = self._error(HTTPException(500)) if 500 not in self.error_handlers else self._to_response(self.error_handlers[500](e))\n                response.status_code = 500\n            for f in self.after_funcs:\n                response = f(response) or response\n            if req.method == "HEAD":\n                response.headers["Content-Length"] = len(response.data)\n                response.data = b""\n            return response\n        finally:\n            _current.reset(token)\n\n    def __call__(self, environ, start_response):\n        response = self.full_dispatch(environ)\n        if "Content-Length" not in response.headers:\n            response.headers["Content-Length"] = len(response.data)\n        start_response(response.status, response.headers.items())\n        return [response.data]\n\n    def test_client(self):\n        return _TestClient(self)\n\n\nclass _MethodNotAllowed(Exception):\n    def __init__(self, allowed):\n        self.allowed = allowed\n\n\nclass Flask(_App):\n    pass\n\n\nclass _TestClient:\n    def __init__(self, app):\n        self.app = app\n        self.cookies = {}\n\n    def open(self, path, method="GET", json=None, data=None, headers=None, query_string=None):\n        url = urlsplit(path)\n        query = url.query\n        if query_string:\n            query = (query + "&" if query else "") + urlencode(query_string, doseq=True)\n        body, ctype = b"", None\n        if json is not None:\n            import json as _json\n            body, ctype = _json.dumps(json).encode("utf-8"), "application/json"\n        elif isinstance(data, dict):\n            body, ctype = urlencode(data, doseq=True).encode("utf-8"), "application/x-www-form-urlencoded"\n        elif data is not None:\n            body = data.encode("utf-8") if isinstance(data, str) else data\n        environ = {"REQUEST_METHOD": method.upper(), "PATH_INFO": url.path or "/", "QUERY_STRING": query,\n                   "SERVER_NAME": "localhost", "SERVER_PORT": "80", "wsgi.url_scheme": "http",\n                   "wsgi.input": io.BytesIO(body), "CONTENT_LENGTH": str(len(body))}\n        headers = dict(headers or {})\n        if ctype and not any(k.lower() == "content-type" for k in headers):\n            environ["CONTENT_TYPE"] = ctype\n        for k, v in headers.items():\n            if k.lower() == "content-type":\n                environ["CONTENT_TYPE"] = v\n            else:\n                environ["HTTP_" + k.upper().replace("-", "_")] = v\n        if self.cookies:\n            environ["HTTP_COOKIE"] = "; ".join(f"{k}={v}" for k, v in self.cookies.items())\n        response = self.app.full_dispatch(environ)\n        for k, v in response.headers.items():\n            if k.lower() == "set-cookie":\n                name, _, rest = v.partition("=")\n                self.cookies[name] = rest.split(";")[0]\n        return response\n\n    def get(self, path, **kw):\n        return self.open(path, "GET", **kw)\n\n    def post(self, path, **kw):\n        return self.open(path, "POST", **kw)\n\n    def put(self, path, **kw):\n        return self.open(path, "PUT", **kw)\n\n    def patch(self, path, **kw):\n        return self.open(path, "PATCH", **kw)\n\n    def delete(self, path, **kw):\n        return self.open(path, "DELETE", **kw)\n')

**Predict before you run:** what will this print?



In [ ]:
env = {"DEBUG": "false", "WORKERS": "4", "DATABASE_URL": "postgresql://db/shop"}

debug = bool(env.get("DEBUG"))
workers = env.get("WORKERS") * 2
print(debug, workers)

debug = env.get("DEBUG", "").lower() in ("1", "true", "yes")
workers = int(env.get("WORKERS", "2")) * 2
print(debug, workers)

In [ ]:
import json, time, uuid
from miniflask import Flask, g, request

app = Flask(__name__)
LOG = []

@app.before_request
def start():
    incoming = request.headers.get("X-Request-Id", "")
    g.request_id = incoming if incoming.isalnum() and len(incoming) <= 64 else uuid.uuid4().hex
    g.started = time.perf_counter()

@app.after_request
def finish(response):
    response.headers["X-Request-Id"] = g.request_id
    LOG.append(json.dumps({"level": "INFO", "msg": "request", "method": request.method, "path": request.path,
                           "status": response.status_code, "request_id": g.request_id}))
    return response

@app.get("/healthz")
def live():
    return {"status": "ok"}

DATABASE_UP = [True]

@app.get("/readyz")
def ready():
    if not DATABASE_UP[0]:
        return {"status": "unavailable", "failing": ["database"]}, 503
    return {"status": "ready"}

client = app.test_client()
print(client.get("/healthz", headers={"X-Request-Id": "abc123"}).headers["X-Request-Id"])
DATABASE_UP[0] = False
print(client.get("/readyz").status_code, client.get("/healthz").status_code)
print(LOG[0])

> **Key idea.** Keep configuration in the environment, parsed and validated at start-up into typed settings, reporting every problem and refusing to start; environment values are strings (bool("false") is True); secrets never in code, images or logs; safe production defaults (DEBUG off, SECRET_KEY required). Run under Gunicorn (WSGI) or Uvicorn (ASGI) with several workers, behind a reverse proxy that terminates TLS and serves static files; trust X-Forwarded-For and -Proto only from your proxies; allow cross-origin calls with specific CORS origins. Offer fast liveness (/healthz) and readiness (/readyz, 503 when dependencies are down) checks; log structured JSON to stdout with a request id per request, echoed in X-Request-Id; watch metrics and alert. Build pinned, non-root container images with dependencies installed before the code; release by rolling (gated on readiness, with rollback), blue-green or canary, shut down gracefully, and keep migrations compatible with old and new code.

### Exercise: Settings from the environment

Write `load_settings(env)`: build a dict of settings from `env` (a dict standing for `os.environ`) following `SCHEMA` (given): each entry is `(name, kind, default)`, where `kind` is `"str"`, `"int"`, `"bool"`, `"list"` (comma-separated, stripped, empty items dropped) or `"url"` (must start with `http://`, `https://`, `postgresql://` or `sqlite://`), and a default of `REQUIRED` means it must be set. Booleans accept `true/false`, `1/0`, `yes/no` and `on/off` in any case. Collect **every** problem as `"NAME: problem"` (`"is required"`, `"must be an integer"`, `"must be true or false"`, `"must be a URL"`) and raise `ConfigError` (given) with them all as `.problems`. Also `safe_repr(settings)`: a string of the settings sorted by name as `NAME=value` pairs joined by `", "`, with any setting whose name contains `KEY`, `SECRET`, `TOKEN` or `PASSWORD` shown as `****`.

In [ ]:
REQUIRED = object()

SCHEMA = [
    ("DEBUG", "bool", False),
    ("PORT", "int", 8000),
    ("WORKERS", "int", 2),
    ("SECRET_KEY", "str", REQUIRED),
    ("DATABASE_URL", "url", REQUIRED),
    ("ALLOWED_HOSTS", "list", ["localhost"]),
    ("LOG_LEVEL", "str", "INFO"),
]

class ConfigError(Exception):
    def __init__(self, problems):
        super().__init__("; ".join(problems))
        self.problems = problems

def load_settings(env):
    ___

def safe_repr(settings):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_settings():\n    \"parsed, validated and masked\"\n    s = load_settings({\"DEBUG\": \"Yes\", \"PORT\": \" 9000 \", \"SECRET_KEY\": \"s3cr3t\", \"DATABASE_URL\": \"postgresql://db/shop\",\n                       \"ALLOWED_HOSTS\": \"shop.example.com, api.example.com,,\"})\n    assert s == {\"DEBUG\": True, \"PORT\": 9000, \"WORKERS\": 2, \"SECRET_KEY\": \"s3cr3t\", \"DATABASE_URL\": \"postgresql://db/shop\",\n                 \"ALLOWED_HOSTS\": [\"shop.example.com\", \"api.example.com\"], \"LOG_LEVEL\": \"INFO\"}\n    assert load_settings({\"DEBUG\": \"false\", \"SECRET_KEY\": \"x\", \"DATABASE_URL\": \"sqlite:///a.db\"})[\"DEBUG\"] is False\n    try:\n        load_settings({\"DEBUG\": \"maybe\", \"PORT\": \"eighty\", \"DATABASE_URL\": \"db.example.com\"})\n        assert False\n    except ConfigError as e:\n        assert e.problems == [\"DEBUG: must be true or false\", \"PORT: must be an integer\", \"SECRET_KEY: is required\", \"DATABASE_URL: must be a URL\"]\n    text = safe_repr(s)\n    assert \"s3cr3t\" not in text and \"SECRET_KEY=****\" in text and text.startswith(\"ALLOWED_HOSTS=\")")

### Exercise: Trusting the proxy, and only the proxy

Write WSGI middleware `proxy_fix(app, trusted, hops=1)`: if the request's `REMOTE_ADDR` is in the set `trusted` (the proxies' addresses), set `REMOTE_ADDR` to the client address from `X-Forwarded-For`, which is a comma-separated list where each proxy appends what it saw: take the entry `hops` places from the **right** (the one your outermost trusted proxy saw), or leave it if the list is too short; and set `wsgi.url_scheme` from `X-Forwarded-Proto` (only `http` or `https`). Keep the originals as `atlas.orig_remote_addr` and `atlas.orig_scheme`. From any other address, change nothing (the headers could be forged).

In [ ]:
def proxy_fix(app, trusted, hops=1):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_proxy():\n    \"forwarded headers\"\n    seen = {}\n    def app(environ, start_response):\n        seen.update(addr=environ[\"REMOTE_ADDR\"], scheme=environ[\"wsgi.url_scheme\"])\n        start_response(\"200 OK\", [])\n        return [b\"\"]\n    fixed = proxy_fix(app, {\"10.0.0.5\"})\n    def call(remote, xff=None, proto=None, middleware=fixed):\n        env = {\"REMOTE_ADDR\": remote, \"wsgi.url_scheme\": \"http\"}\n        if xff:\n            env[\"HTTP_X_FORWARDED_FOR\"] = xff\n        if proto:\n            env[\"HTTP_X_FORWARDED_PROTO\"] = proto\n        middleware(env, lambda s, h: None)\n        return seen[\"addr\"], seen[\"scheme\"], env\n    addr, scheme, env = call(\"10.0.0.5\", \"203.0.113.9\", \"https\")\n    assert (addr, scheme) == (\"203.0.113.9\", \"https\") and env[\"atlas.orig_remote_addr\"] == \"10.0.0.5\"\n    assert call(\"10.0.0.5\", \"1.2.3.4, 203.0.113.9\")[0] == \"203.0.113.9\", \"a client can put anything on the left\"\n    assert call(\"198.51.100.7\", \"1.1.1.1\", \"https\")[:2] == (\"198.51.100.7\", \"http\"), \"not from our proxy: ignore the headers\"\n    assert call(\"10.0.0.5\", \"203.0.113.9\", \"gopher\")[1] == \"http\"\n    two = proxy_fix(app, {\"10.0.0.5\"}, hops=2)\n    assert call(\"10.0.0.5\", \"fake, 203.0.113.9, 10.0.0.9\", middleware=two)[0] == \"203.0.113.9\"\n    assert call(\"10.0.0.5\", \"203.0.113.9\", middleware=two)[0] == \"10.0.0.5\"")

### Exercise: Liveness and readiness

Write `add_health_checks(app, checks)`: register `GET /healthz` (always `{"status": "ok"}`) and `GET /readyz`, which runs every check in `checks` (a dict of name to a function returning `True` when that dependency is fine; a check that **raises** counts as failing) and answers `200` with `{"status": "ready", "checks": {name: "ok", ...}}`, or `503` with `"status": "unavailable"` and each failing check as `"failing"` (without the exception's message, which might leak details), and a `Retry-After: 5` header. Both answer with `Cache-Control: no-store`.

In [ ]:
from miniflask import Flask

def add_health_checks(app, checks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_health():\n    \"probes for the platform\"\n    from miniflask import Flask\n    state = {\"db\": True}\n    def cache():\n        raise ConnectionError(\"cache.internal:6379 refused\")\n    app = add_health_checks(Flask(__name__), {\"database\": lambda: state[\"db\"]})\n    client = app.test_client()\n    assert client.get(\"/healthz\").get_json() == {\"status\": \"ok\"} and client.get(\"/healthz\").headers[\"Cache-Control\"] == \"no-store\"\n    r = client.get(\"/readyz\")\n    assert r.status_code == 200 and r.get_json() == {\"status\": \"ready\", \"checks\": {\"database\": \"ok\"}}\n    state[\"db\"] = False\n    r = client.get(\"/readyz\")\n    assert r.status_code == 503 and r.get_json() == {\"status\": \"unavailable\", \"checks\": {\"database\": \"failing\"}} and r.headers[\"Retry-After\"] == \"5\"\n    app2 = add_health_checks(Flask(\"b\"), {\"database\": lambda: True, \"cache\": cache})\n    r = app2.test_client().get(\"/readyz\")\n    assert r.status_code == 503 and r.get_json()[\"checks\"] == {\"database\": \"ok\", \"cache\": \"failing\"} and \"6379\" not in r.text\n    assert app2.test_client().get(\"/healthz\").status_code == 200")

### Exercise: Structured logs and request ids

Write `request_logging(app, log, clock, new_id)` for a miniflask app: a `before_request` hook that takes the request id from `X-Request-Id` if it's 8 to 64 characters of letters, digits and `-`, and otherwise uses `new_id()`, storing it and the start time (`clock()`) in `g`; and an `after_request` hook that adds `X-Request-Id` to the response and appends to `log` one JSON line (compact, keys in this order): `{"level": ..., "msg": "request", "method", "path", "status", "ms": whole milliseconds, "request_id"}`, with level `"ERROR"` for 5xx, `"WARNING"` for 4xx, else `"INFO"`. The path is logged **without** its query string (which may hold personal data). Return the app.

In [ ]:
import json, re
from miniflask import Flask, g, request

def request_logging(app, log, clock, new_id):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_logging():\n    \"one JSON line per request\"\n    import json\n    from miniflask import Flask\n    times = iter([1.0, 1.0125, 2.0, 2.3])\n    ids = iter([\"generated-1\", \"generated-2\"])\n    log = []\n    app = Flask(__name__)\n    @app.get(\"/orders\")\n    def orders():\n        return {\"orders\": []}\n    request_logging(app, log, lambda: next(times), lambda: next(ids))\n    client = app.test_client()\n    r = client.get(\"/orders?email=ada@example.com\", headers={\"X-Request-Id\": \"abc-12345\"})\n    assert r.headers[\"X-Request-Id\"] == \"abc-12345\"\n    assert log[0] == '{\"level\":\"INFO\",\"msg\":\"request\",\"method\":\"GET\",\"path\":\"/orders\",\"status\":200,\"ms\":12,\"request_id\":\"abc-12345\"}'\n    r = client.get(\"/missing\", headers={\"X-Request-Id\": \"<script>\"})\n    entry = json.loads(log[1])\n    assert r.headers[\"X-Request-Id\"] == \"generated-1\" and entry[\"level\"] == \"WARNING\" and entry[\"ms\"] == 300")

### Exercise: Reviewing a Dockerfile

Write `review_dockerfile(text)`: a list of problems (in this order, each at most once): `"base image not pinned"` if the first `FROM` has no tag or the tag `latest`; `"secret in ENV: NAME"` for each `ENV NAME=...` (or `ENV NAME value`) whose name contains `KEY`, `SECRET`, `TOKEN` or `PASSWORD`; `"code copied before dependencies are installed"` if a `COPY . ` line comes before the first `RUN pip install`; `"runs as root"` if there's no `USER` line (or only `USER root`); and `"no CMD"` if there's no `CMD` or `ENTRYPOINT`. Instructions are case-insensitive; ignore blank lines and comments.

In [ ]:
import re

def review_dockerfile(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_dockerfile():\n    \"container habits\"\n    good = \"FROM python:3.13-slim\\nWORKDIR /app\\nCOPY requirements.txt .\\nRUN pip install -r requirements.txt\\nCOPY . .\\nUSER app\\nCMD [\\\"gunicorn\\\", \\\"app:app\\\"]\\n\"\n    assert review_dockerfile(good) == []\n    bad = \"# my image\\nfrom python\\nenv SECRET_KEY=abc123\\nENV API_TOKEN xyz\\nENV DEBUG=0\\nCOPY . .\\nRUN pip install -r requirements.txt\\nUSER root\\n\"\n    assert review_dockerfile(bad) == [\"base image not pinned\", \"secret in ENV: SECRET_KEY\", \"secret in ENV: API_TOKEN\",\n                                      \"code copied before dependencies are installed\", \"runs as root\", \"no CMD\"]\n    assert review_dockerfile(\"FROM python:latest\\nUSER app\\nENTRYPOINT [\\\"x\\\"]\\n\") == [\"base image not pinned\"]")

### Exercise: How many workers, how many servers?

Write `workers(cores)`: the common starting point `2 * cores + 1`. Then `throughput(workers, threads, avg_ms)`: requests per second one instance can handle if each worker thread handles one request at a time taking `avg_ms` milliseconds on average (rounded down). Finally `instances_needed(peak_rps, per_instance, headroom=0.3, minimum=2)`: enough instances to serve the peak with `headroom` spare (so each runs at most `1 - headroom` of its capacity), rounded up, and never fewer than `minimum` (so one can fail or be replaced during a deploy).

In [ ]:
import math

def workers(cores):
    ___

def throughput(workers, threads, avg_ms):
    ___

def instances_needed(peak_rps, per_instance, headroom=0.3, minimum=2):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_capacity():\n    \"back-of-the-envelope sizing\"\n    assert workers(1) == 3 and workers(4) == 9\n    assert throughput(9, 1, 50) == 180 and throughput(4, 4, 120) == 133\n    assert instances_needed(1000, 180) == 8 and instances_needed(50, 180) == 2 and instances_needed(126, 180) == 2\n    assert instances_needed(253, 180) == 3 and instances_needed(1000, 180, headroom=0, minimum=1) == 6")

### Bug hunt: Debug on, by accident

The environment says `DEBUG=false`, but `bool()` of any non-empty string is `True`, so the app starts in debug mode and this prints `debug: True`. Parse the text properly, so it prints `debug: False`.

In [ ]:
env = {"DEBUG": "false"}
debug = bool(env.get("DEBUG", ""))
print("debug:", debug)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"strings aren't booleans\"\n    expect_output(\"debug: False\")")

### Bug hunt: A secret in the logs

The start-up message logs the whole settings dict, secret key included, and logs are read by many people and kept for a long time: this prints the key. Mask any setting whose name contains `KEY`, `SECRET`, `TOKEN` or `PASSWORD`, so this prints `starting with {'DEBUG': False, 'SECRET_KEY': '****', 'PORT': 8000}`.

In [ ]:
settings = {"DEBUG": False, "SECRET_KEY": "8f3a-very-secret", "PORT": 8000}
print("starting with", settings)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"masked secrets\"\n    expect_output(\"starting with {'DEBUG': False, 'SECRET_KEY': '****', 'PORT': 8000}\")")

### Bug hunt: Ready, while the database is down

The readiness check always answers `200`, so the load balancer keeps sending traffic to an instance that can't reach its database, and this prints `200`. Check the database, answering `503` when it's down, so this prints `503`.

In [ ]:
from miniflask import Flask

app = Flask(__name__)
database_up = False

@app.get("/readyz")
def ready():
    return {"status": "ready"}

print(app.test_client().get("/readyz").status_code)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"readiness means ready\"\n    expect_output(\"503\")")

### Puzzle: A Dockerfile

Arrange the lines of a Dockerfile that caches dependencies, runs as a normal user and starts Gunicorn. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # bool("0")
q = ___   # the starting number of Gunicorn workers for 2 cores, by the usual rule
r = ___   # the status a readiness check gives while a dependency is down
s = ___   # True or False: an app should trust X-Forwarded-For from any client
t = ___   # where twelve-factor apps write their logs: "a file" or "standard output"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is True and q == 5 and r == 503, \"p, q and r\"\n    assert s is False and t == \"standard output\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a rolling deployment

Simulate a rolling deployment. `rolling_deploy(instances, new_version, start, ready, batch=1, min_healthy=None)`: `instances` is a list of the current instances (dicts with `"name"` and `"version"`); `start(name, version)` starts a replacement instance and returns its name; `ready(name)` returns whether it passed its readiness check. Replace the old instances `batch` at a time, in order: for each, **start** a new instance (named by `start`), and if it's ready, **stop** the old one (log `"stop OLD"`), otherwise **stop the new one**, stop deploying, and **roll back**: every new instance started so far that's still running is stopped, and for each old instance already stopped, a replacement running the **old** version is started (and assumed ready), in order. Never let the number of ready instances drop below `min_healthy` (default: the original count): new instances are always started before old ones are stopped. Return `(succeeded, running, log)`: whether the deploy finished, the list of `(name, version)` running at the end, and the log of actions (`"start NAME VERSION"`, `"ready NAME"`, `"not ready NAME"`, `"stop NAME"`, `"rollback"`).

In [ ]:
def rolling_deploy(instances, new_version, start, ready, batch=1, min_healthy=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rolling():\n    \"replace, check, and roll back\"\n    counter = [0]\n    def start(name, version):\n        counter[0] += 1\n        return f\"web-{counter[0] + 3}\"\n    old = [{\"name\": \"web-1\", \"version\": \"1.4\"}, {\"name\": \"web-2\", \"version\": \"1.4\"}, {\"name\": \"web-3\", \"version\": \"1.4\"}]\n    ok, running, log = rolling_deploy(old, \"1.5\", start, ready=lambda name: True)\n    assert ok and running == [(\"web-4\", \"1.5\"), (\"web-5\", \"1.5\"), (\"web-6\", \"1.5\")]\n    assert log[:4] == [\"start web-4 1.5\", \"ready web-4\", \"stop web-1\", \"start web-5 1.5\"] and log.count(\"stop web-3\") == 1\n    counter[0] = 0\n    ok, running, log = rolling_deploy(old, \"1.5\", start, ready=lambda name: name != \"web-5\")\n    assert not ok and sorted(running) == [(\"web-2\", \"1.4\"), (\"web-3\", \"1.4\"), (\"web-6\", \"1.4\")]\n    assert log == [\"start web-4 1.5\", \"ready web-4\", \"stop web-1\", \"start web-5 1.5\", \"not ready web-5\", \"stop web-5\",\n                   \"rollback\", \"stop web-4\", \"start web-6 1.4\"]\n    counter[0] = 0\n    ok, running, log = rolling_deploy(old, \"1.5\", start, ready=lambda name: True, batch=2)\n    assert ok and len(running) == 3 and log[:6] == [\"start web-4 1.5\", \"ready web-4\", \"start web-5 1.5\", \"ready web-5\", \"stop web-1\", \"stop web-2\"]")

## Warm-up

Environment variables are **always strings**: `bool("false")` is `True` (any non-empty string is), and `"4" * 2` is `"44"`. Configuration from the environment is the standard way to deploy, and every value needs parsing and checking, once, at start-up, so a typo stops the app from starting rather than misbehaving in production.

## Configuration

The **twelve-factor app** guidelines, widely followed for web services, say: keep **configuration in the environment**, separate from the code. The same build runs in development, staging and production; only the environment differs.

- Read settings from **environment variables** (`os.environ`), perhaps loaded from a `.env` file in development (never committed: Week 79). Typical ones: `DATABASE_URL`, `SECRET_KEY`, `DEBUG`, `ALLOWED_HOSTS`, `LOG_LEVEL`, `PORT`.
- **Parse and validate** them at start-up into a typed settings object: integers, booleans (`true`, `1`, `yes`), lists (comma-separated), URLs. Report **every** problem at once and refuse to start. Libraries such as `pydantic-settings` do this from type hints, like FastAPI.
- **Secrets** (keys, passwords, tokens) come from the environment or a secrets manager, never from code or the repository, and must never appear in logs or error pages: show them masked (`****`).
- **Production defaults** must be safe: `DEBUG` off, `SECRET_KEY` required with no default, HTTPS only. A debugger page in production lets anyone run code on your server.

## Servers and proxies

Development servers (`flask run`, `fastapi dev`) are for one person. In production:

- An application server runs your app with several **workers**: **Gunicorn** for WSGI (`gunicorn "shop:create_app()" --workers 4 --bind 0.0.0.0:8000`), **Uvicorn** for ASGI (often as Gunicorn workers). A common starting point is `2 x cores + 1` workers for synchronous apps; measure and adjust.
- A **reverse proxy** such as **nginx** or Caddy (or a cloud load balancer) sits in front: it terminates **TLS** (HTTPS, with certificates from Let's Encrypt), serves static files, buffers slow clients, limits request sizes, and spreads requests over several app servers.
- Behind a proxy, the app sees the **proxy's** address and plain `http`. The proxy passes the originals in headers: **`X-Forwarded-For`** (the client's address, with each proxy appending the one it got the request from), **`X-Forwarded-Proto`** (`https`) and `X-Forwarded-Host` (or the standard `Forwarded` header). The app must trust these **only from its own proxies**, because anyone can send them; Werkzeug's `ProxyFix` middleware does this for Flask. Then redirects, URLs and rate limits use the real client and scheme.
- Browsers enforce the **same-origin policy**: a page from `https://shop.example.com` can only call `https://api.example.com` if the API sends **CORS** headers (`Access-Control-Allow-Origin: https://shop.example.com`), after answering a **preflight** `OPTIONS` request for anything but the simplest calls. Allow specific origins, never `*` with credentials.

## Health, logs and metrics

Once deployed, you need to know it's working without logging in to look:

- **Health checks.** A **liveness** endpoint (`/healthz`) answers `200` if the process is running at all; the platform restarts the instance if it stops answering. A **readiness** endpoint (`/readyz`) answers `200` only when the instance can really serve (its database and other dependencies are reachable), and `503` otherwise, so the load balancer sends it no traffic until it's ready. Keep both fast and unauthenticated, and don't let them leak details.
- **Logs.** Write logs to **standard output** (the platform collects them), one **structured** JSON object per line (`{"ts": "...", "level": "INFO", "msg": "request", "method": "GET", "path": "/orders", "status": 200, "ms": 12, "request_id": "..."}`), so they can be searched and counted. Give every request a **request id** (taken from an incoming `X-Request-Id` if it's well-formed, otherwise generated), include it in every log line and send it back in the response, so one request can be followed through every service it touches.
- **Metrics** (requests per second, error rate, latency percentiles, saturation) are counted and graphed by tools such as Prometheus and Grafana, with **alerts** when they cross thresholds (Week 85 has more).

## Containers and releases

A **container image** packages the app with its exact Python version and dependencies, so it runs the same everywhere. A typical `Dockerfile`:

```dockerfile
FROM python:3.13-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
RUN useradd --create-home app
USER app
EXPOSE 8000
CMD ["gunicorn", "shop:create_app()", "--bind", "0.0.0.0:8000", "--workers", "4"]
```

- Pin the base image to a specific version, never `latest`.
- Copy and install the requirements (the lock file, Week 73 Day 5) **before** copying the code, so rebuilding after a code change reuses the cached dependency layer.
- Don't run as root; don't bake secrets into the image (pass them as environment variables when it runs); keep the image small (`-slim`, no build tools left behind).

**Releasing** a new version without downtime:

- A **rolling** deployment replaces instances one (or a few) at a time: start a new one, wait for it to pass its **readiness** check, move traffic to it, then stop an old one; if new instances never become ready, stop and **roll back**.
- **Blue-green** runs the whole new version beside the old and switches traffic at once (and back, if needed); **canary** releases send a small share of traffic to the new version first and watch its errors.
- Stopping an instance should be **graceful**: stop accepting new requests, finish the ones in progress, then exit.
- **Database migrations** must work with both the old and new code during the changeover: add a column first, deploy code that uses it, and remove old columns only in a later release.

## Practice

## On your own computer

Run an app the production way, locally:

```bash
pip install gunicorn
```

```bash
SECRET_KEY=dev-only DATABASE_URL=sqlite:///shop.db gunicorn "shop:create_app()" --workers 3 --bind 127.0.0.1:8000
```

Then build and run it in a container (with Docker or Podman installed):

```bash
docker build -t shop:1.0 . && docker run --rm -p 8000:8000 -e SECRET_KEY=dev-only -e DATABASE_URL=sqlite:///shop.db shop:1.0
```

For a real deployment, platforms that run containers or Python apps from a Git repository (and give you HTTPS, logs and health checks) are the quickest start; a virtual server with nginx, Gunicorn and a `systemd` service teaches you the most.

## Quiz

## Daily challenge

## Before you go

Write down how you'd deploy one of this week's apps: where it runs, how it gets its settings and secrets, what's in front of it, how you'd know it's healthy, and how you'd roll back. Tomorrow's Forge builds a complete small backend in one go, ready for exactly that.